<h1 style="color:#D30982;text-align:center;">Chapter 8: What's Next</h1>

<h2 style="color:#D30982;">What you built</h2>

You have now completed the full Microsoft QDK Chemistry pipeline — from loading a molecular structure all the way to extracting a ground-state energy via quantum phase estimation on a simulator. Here is the complete arc:

| Chapter | Topic | What you did |
|---|---|---|
| 0 | Getting Started | Set up the QDK, ran a Q# project locally |
| 1 | Molecular Input | Loaded structures, explored `Structure`, `Orbitals`, `Wavefunction` |
| 2 | Classical Functionality | Ran SCF, Pipek-Mezey localization, stability checking |
| 3 | Active Space Selection | Applied valence, autoCAS, AVAS, and autoCAS-EOS selectors |
| 4 | Hamiltonian & Qubit Mapping | Built the molecular Hamiltonian and mapped it to qubits via Jordan-Wigner |
| 5 | State Preparation | Synthesized isometry circuits; compared sparse vs. dense methods |
| 6 | Quantum Phase Estimation | Ran end-to-end IQPE; swept phase bits toward chemical accuracy |
| 7 | Plugins & Extension | Built and registered a custom `ActiveSpaceSelector` using the QDK plugin system |

The system you worked with — the QDK Chemistry library — is designed to scale to real Magne hardware as it becomes available. The skills you developed here are directly transferable to production quantum chemistry workflows.

<h2 style="color:#D30982;">This is a living course</h2>

This course is designed to iterate based on feedback from researchers like you. What you found confusing, what was missing, and what was most useful directly shapes the next version. The sections below outline how to stay involved, file bugs, suggest features, and optionally go deeper with challenge projects.

<h2 style="color:#D30982;">Part 1: Your Feedback</h2>

Please share your experience with the course. Your response goes directly to the QDK team and informs the next iteration.

Useful things to address:
- Which chapters or sections felt unclear or under-explained?
- Were there workflows or algorithms you expected to see but didn't?
- What was the most valuable thing you learned?
- Any bugs, errors, or broken cells you encountered?

In [ ]:
# Work with Microsoft and QuNorth to have feedback here in viable format. 

<h2 style="color:#D30982;">Part 2: Share Your Workflow</h2>

If you developed a workflow during or after this course — a modified pipeline, a new molecule, a custom plugin, a noise simulation — we want to hear about it.

The QDK community is building a library of **GitHub Copilot workflow templates**: short, annotated notebooks that capture real workflows researchers have found useful. If you have something worth sharing:

1. **Fork** `github.com/microsoft/qdk-chemistry` and commit your notebook to the `examples/` directory
2. **Open a pull request** with a short description of what problem it solves
3. Or **share it in the community channel** for feedback before formalizing it

What did you build, modify, or discover during the course?

# Question: ch08-q2

> Question not found (HTTP 404)


<h2 style="color:#D30982;">Part 3: Post-Course Resources</h2>

**GitHub — primary hub for the QDK ecosystem:**
- <a href="https://github.com/microsoft/qdk-chemistry" target="_blank">`github.com/microsoft/qdk-chemistry`</a> — the library you used in this course; source, docs, and examples
- <a href="https://github.com/microsoft/qdk" target="_blank">`github.com/microsoft/qdk`</a> — Q# compiler, language service, VS Code extension, and standard library

**Primary paper:**
- <a href="https://arxiv.org/abs/2601.15253" target="_blank">Baker et al. (2025) — "QDK/Chemistry: A Modular Toolkit for Quantum Chemistry Applications"</a> — the design and architecture paper for the toolkit underlying this course

**Filing bugs — fastest path to resolution:**

If you hit a bug, a missing feature, or incorrect behavior during your research, filing a GitHub issue is the fastest way to get it resolved. The QDK team triages chemistry issues directly from `github.com/microsoft/qdk-chemistry/issues`. When filing:
- Include your QDK Chemistry version (`pip show qdk-chemistry`)
- Include a minimal reproducible example — even 5 lines of code that trigger the bug
- Describe the expected vs. actual behavior

**Feature requests — your input shapes the roadmap:**

If you developed a real application and hit a gap in the library — a missing algorithm, an inflexible API, a needed backend — open a GitHub Discussion or issue tagged `enhancement`. The QDK team tracks research-driven requests and prioritizes based on real use cases. Your work with Magne is exactly the feedback that shapes what gets built next.

**Community:**
- QuNorth Quantum Discord: [TALK TO QUNORTH AND MAKE NORDIC COMMUNITY DISCORD]
- <a href="https://www.microsoft.com/en-us/research/research-area/quantum-computing/" target="_blank">Microsoft Research — Quantum</a>

<h2 style="color:#D30982;">Part 4: Challenge Projects</h2>

These optional projects are designed to push beyond the course material into genuine research problems. Each one builds directly on the skills and tools from chapters 0–7.

---

<h3>Challenge 1: Noise-aware QPE</h3>

**Goal:** Determine the 2-qubit gate error rate at which IQPE on the N₂ autoCAS-EOS active space can no longer achieve chemical accuracy (1.6 mHa).

**Starting point:** Use the ch06 IQPE pipeline with `num_bits=10`, `shots_per_bit=10`. Replace `qdk_sparse_state_simulator` with `qiskit_aer_simulator` and attach a `QuantumErrorProfile` depolarizing noise model.

```python
from qdk_chemistry.data import QuantumErrorProfile
noise = QuantumErrorProfile(
    name="depolarizing",
    description="Uniform depolarizing noise",
    errors={
        "cx": {"type": "depolarizing_error", "rate": 1e-3, "num_qubits": 2},
        "rz": {"type": "depolarizing_error", "rate": 1e-4, "num_qubits": 1},
    },
)
executor = create("circuit_executor", "qiskit_aer_simulator")
# pass noise_model=noise to pe.run(...)
```

Sweep `cx` error rates across `[1e-4, 5e-4, 1e-3, 5e-3, 1e-2]` and plot QPE energy error vs. error rate. At what rate does noise dominate over Trotter error?

---

<h3>Challenge 2: Extend the pipeline to water</h3>

**Goal:** Apply the full ch00–ch07 pipeline to water (H₂O) and compare its resource profile against N₂.

**Starting point:** `../examples/data/water.structure.xyz` is already in the repo. Run the same SCF → valence → MP2 → autoCAS-EOS → Hamiltonian → IQPE pipeline. Answer:
- How many active orbitals does autoCAS-EOS select for H₂O vs N₂?
- How many Pauli terms does the H₂O qubit Hamiltonian have? How does the Schatten norm compare?
- What IQPE circuit depth and number of phase bits are needed to reach 1.6 mHa accuracy?
- Try `../examples/data/benzene_diradical.structure.xyz` for a larger challenge.

---

<h3>Challenge 3: Build a new domain plugin</h3>

**Goal:** Extend the QDK Chemistry library with a new backend using the ch07 plugin pattern.

Choose one of:

**Option A — New active space selector:** Build a selector that picks orbitals by their contribution to the MP2 correlation energy (rather than occupation or orbital energy). Hint: correlation energy contributions are proportional to the magnitude of MP2 amplitudes, which you can extract from the two-electron integrals and the orbital energy denominator.

**Option B — New SCF solver wrapper:** Wrap an external chemistry package (e.g., Psi4, ORCA, or any package with a Python API) as a `ScfSolver` subclass. Register it with the QDK so that `create("scf_solver", "my_backend")` returns your wrapper. The rest of the pipeline should then work unchanged.

**Option C — New time evolution builder:** Implement a randomized compiler (e.g., a simple version of qDRIFT with a custom importance-sampling scheme) as a `TimeEvolutionBuilder` subclass. Compare its step-term count and QPE accuracy against the built-in `partially_randomized` builder from ch06.

<h2 style="color:#D30982;">Thank you</h2>

This course was built to put the full Microsoft quantum chemistry stack in the hands of people who can actually stress-test it. Your feedback, bugs, feature requests, and research results are what make the next version better, and what directly shape how the QDK evolves toward <a href="https://www.nature.com/articles/s41467-023-37587-6" target="_blank">fault-tolerant quantum chemistry</a> on Magne.

Let's shape the future of the Nordic quantum ecosystem!